In [ ]:
#@title
from IPython.display import display, HTML
display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
  .oxrna-intro {
    font-family: 'Inter', sans-serif;
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 60%, #0f4c75 100%);
    border-radius: 16px;
    padding: 40px 48px;
    color: #f1f5f9;
    max-width: 820px;
    margin: 12px auto;
    box-shadow: 0 20px 60px rgba(0,0,0,0.4);
  }
  .oxrna-intro .badge {
    display: inline-block;
    background: rgba(56,189,248,0.15);
    border: 1px solid rgba(56,189,248,0.4);
    color: #38bdf8;
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 2px;
    text-transform: uppercase;
    padding: 4px 12px;
    border-radius: 20px;
    margin-bottom: 18px;
  }
  .oxrna-intro h1 {
    font-size: 32px; font-weight: 700; margin: 0 0 10px;
    color: #f8fafc; letter-spacing: -0.5px;
  }
  .oxrna-intro .sub {
    font-size: 15px; color: #94a3b8; margin-bottom: 28px; line-height: 1.6;
  }
  .oxrna-intro .steps { display: flex; gap: 14px; flex-wrap: wrap; }
  .oxrna-intro .step {
    background: rgba(255,255,255,0.06);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 12px; padding: 16px 20px; flex: 1; min-width: 160px;
  }
  .oxrna-intro .step .num {
    font-size: 11px; font-weight: 700; color: #38bdf8;
    letter-spacing: 1.5px; text-transform: uppercase; margin-bottom: 6px;
  }
  .oxrna-intro .step p { font-size: 13px; color: #cbd5e1; margin: 0; line-height: 1.5; }
  .oxrna-intro hr { border: none; border-top: 1px solid rgba(255,255,255,0.08); margin: 28px 0; }
  .oxrna-intro .note { font-size: 12.5px; color: #64748b; line-height: 1.6; }
  .oxrna-intro .note code {
    background: rgba(255,255,255,0.08); padding: 1px 6px; border-radius: 4px;
    font-family: monospace; color: #7dd3fc; font-size: 12px;
  }
</style>
<div class='oxrna-intro'>
  <div class='badge'>oxRNA Setup Wizard</div>
  <h1>PDB &rarr; oxRNA Input Generator</h1>
  <p class='sub'>Provide a PDB file path and this notebook will validate whether it
  contains RNA, then generate the exact inputs needed for an oxRNA simulation
  &mdash; including the <code>.dat</code> and <code>.top</code> output filenames.</p>
  <div class='steps'>
    <div class='step'><div class='num'>Step 1</div><p>Enter your <code>.pdb</code> file path in any format</p></div>
    <div class='step'><div class='num'>Step 2</div><p>Choose strand direction and options</p></div>
    <div class='step'><div class='num'>Step 3</div><p>Click <strong>Configure</strong> &mdash; outputs appear below</p></div>
  </div>
  <hr>
  <p class='note'>Accepts any path format: <code>/content/rna.pdb</code>, <code>r&quot;C:\\data\\rna.pdb&quot;</code>,
  single or double quotes, raw strings. Run both cells top to bottom before using.</p>
</div>
"""))

In [ ]:
#@title
import re, os
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

RNA_RESIDUES = {
    'A','C','G','U','RA','RC','RG','RU',
    'ADE','CYT','GUA','URA','+A','+C','+G','+U'
}
DNA_RESIDUES = {'DA','DC','DG','DT','T'}
SOLVENT     = {'HOH','WAT','MG','ZN','NA','CL','K','MN','CA','FE'}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
.oxw{font-family:'Inter',sans-serif;max-width:820px;margin:12px auto;}
.oxcard{background:#1e293b;border:1px solid #334155;border-radius:14px;
  padding:26px 30px;margin-bottom:0;box-shadow:0 4px 24px rgba(0,0,0,.35);}
.oxlabel{font-size:11px;font-weight:700;color:#64748b;letter-spacing:1.2px;
  text-transform:uppercase;display:block;margin-bottom:6px;}
.result-box{font-family:'Inter',sans-serif;max-width:820px;margin:14px auto;
  background:#1e293b;border-radius:14px;padding:28px 32px;
  border:1px solid #334155;box-shadow:0 4px 24px rgba(0,0,0,.3);}
.tag{display:inline-block;font-size:11px;font-weight:700;letter-spacing:1.5px;
  text-transform:uppercase;padding:3px 11px;border-radius:20px;margin-bottom:12px;}
.tok{background:rgba(34,197,94,.12);border:1px solid rgba(34,197,94,.35);color:#4ade80;}
.twarn{background:rgba(251,191,36,.1);border:1px solid rgba(251,191,36,.35);color:#fbbf24;}
.terr{background:rgba(239,68,68,.1);border:1px solid rgba(239,68,68,.35);color:#f87171;}
.result-box h2{font-size:19px;font-weight:700;color:#f1f5f9;margin:0 0 7px;}
.result-box .msg{font-size:13.5px;color:#94a3b8;margin-bottom:20px;line-height:1.65;}
.cmd-block{background:#0f172a;border:1px solid #1e3a5f;border-radius:10px;padding:18px 22px;}
.cmd-block .clabel{font-size:11px;font-weight:700;color:#38bdf8;
  letter-spacing:1.5px;text-transform:uppercase;margin-bottom:12px;display:block;}
.arow{display:flex;align-items:center;gap:10px;margin-bottom:8px;flex-wrap:wrap;}
.akey{font-size:12px;color:#64748b;min-width:200px;font-family:monospace;}
.aval{font-size:13px;color:#7dd3fc;font-family:monospace;
  background:rgba(56,189,248,.08);padding:3px 10px;border-radius:5px;word-break:break-all;}
.fcmd{margin-top:14px;padding-top:14px;border-top:1px solid #1e3a5f;
  font-family:monospace;font-size:13px;color:#a5f3fc;word-break:break-all;line-height:1.6;}
.fcmd .fl{color:#475569;font-size:11px;margin-bottom:4px;display:block;}
.sbanner{margin-top:20px;background:rgba(34,197,94,.07);
  border:1px solid rgba(34,197,94,.22);border-radius:10px;
  padding:16px 20px;display:flex;align-items:center;gap:12px;}
.sbanner .si{font-size:20px;}
.sbanner .st{font-size:15px;font-weight:700;color:#4ade80;}
</style>
"""

def clean_path(raw):
    s = raw.strip()
    m = re.match(r'^[rR]?(["\'])(.+)\1$', s)
    if m:
        s = m.group(2)
    return s.replace('\\\\', os.sep).replace('\\', os.sep)

def check_pdb(path):
    rna, dna, other = set(), set(), set()
    try:
        with open(path) as f:
            for line in f:
                if line.startswith(('ATOM','HETATM')):
                    res = line[17:20].strip().upper()
                    if res in RNA_RESIDUES:
                        rna.add(res)
                    elif res in DNA_RESIDUES or (len(res)==2 and res[0]=='D'):
                        dna.add(res)
                    elif res not in SOLVENT:
                        other.add(res)
    except Exception as e:
        return None, str(e)
    return {'rna':rna,'dna':dna,'other':other}, None

path_input = widgets.Text(
    placeholder='e.g.  /content/rna.pdb  or  r"C:\\data\\rna.pdb"',
    layout=widgets.Layout(width='100%'),
    style={'description_width':'0px'}
)
dir_select = widgets.Dropdown(
    options=[('3\u2032 \u2192 5\u2032  (flag: 35)', '35'),
             ('5\u2032 \u2192 3\u2032  (flag: 53)', '53')],
    layout=widgets.Layout(width='280px')
)
models_chk = widgets.Checkbox(
    description='Treat MODEL records as separate strands  (-m)',
    value=False, indent=False,
    layout=widgets.Layout(width='auto'),
    style={'description_width':'auto'}
)
btn = widgets.Button(
    description='\u25b6  Configure',
    button_style='',
    layout=widgets.Layout(width='160px', height='40px')
)
btn.style.button_color = '#0ea5e9'
btn.style.font_weight  = '700'

out = widgets.Output()

display(HTML(CSS + """
<div class='oxw'><div class='oxcard'>
  <span class='oxlabel'>PDB File Path</span>
</div></div>
"""))
display(widgets.VBox([
    path_input,
    widgets.HBox([dir_select, models_chk],
                 layout=widgets.Layout(margin='10px 0 4px', gap='24px', align_items='center')),
    btn
], layout=widgets.Layout(padding='0 0 6px')))
display(out)

def on_configure(b):
    path       = clean_path(path_input.value)
    direction  = dir_select.value
    use_models = models_chk.value

    with out:
        clear_output(wait=True)

        if not path:
            display(HTML(CSS + "<div class='result-box'><span class='tag terr'>Error</span>"
                         "<h2>No path provided</h2>"
                         "<p class='msg'>Please enter a .pdb file path above.</p></div>"))
            return

        if not os.path.isfile(path):
            display(HTML(CSS + f"<div class='result-box'><span class='tag terr'>File Not Found</span>"
                         f"<h2>Cannot open file</h2>"
                         f"<p class='msg'>No file found at:<br>"
                         f"<code style='color:#f87171;font-family:monospace'>{path}</code><br><br>"
                         f"Check the path and try again.</p></div>"))
            return

        info, err = check_pdb(path)
        if err:
            display(HTML(CSS + f"<div class='result-box'><span class='tag terr'>Read Error</span>"
                         f"<h2>Could not parse PDB</h2><p class='msg'>{err}</p></div>"))
            return

        basename  = os.path.basename(path)
        top_file  = basename + ".top"
        dat_file  = basename + ".dat"
        oxdna_file= basename + ".oxdna"

        models_arg = " -m" if use_models else ""
        full_cmd   = f'python pdb_oxDNA.py "{path}" {direction}{models_arg}'

        has_rna  = bool(info['rna'])
        has_dna  = bool(info['dna'])

        def fmt(s): return '</code>, <code>'.join(sorted(s))

        if has_dna and not has_rna:
            tag   = "<span class='tag twarn'>Not RNA &#8212; DNA Detected</span>"
            title = "This structure appears to be DNA, not RNA"
            msg   = (f"Residues found: <code>{fmt(info['dna'])}</code>. "
                     f"These are DNA nucleotides. oxRNA is designed for RNA. "
                     f"For DNA simulations use <strong>oxDNA</strong> instead. "
                     f"If the file is meant to be RNA, verify residue naming in the PDB.")
            banner = ""
        elif not has_rna and not has_dna:
            tag   = "<span class='tag twarn'>No Nucleic Acid Found</span>"
            title = "No standard RNA or DNA residues detected"
            other_str = fmt(info['other']) if info['other'] else 'none'
            msg   = (f"Residues seen: <code>{other_str}</code>. "
                     f"The file may contain only protein, ions, or solvent, "
                     f"or residue names may be non-standard.")
            banner = ""
        elif has_rna and has_dna:
            tag   = "<span class='tag twarn'>Mixed RNA &amp; DNA</span>"
            title = "Structure contains both RNA and DNA residues"
            msg   = (f"RNA: <code>{fmt(info['rna'])}</code> &nbsp;|&nbsp; "
                     f"DNA: <code>{fmt(info['dna'])}</code>. "
                     f"oxRNA is intended for pure RNA &#8212; mixed structures may give unreliable results.")
            banner = ""
        else:
            tag   = "<span class='tag tok'>RNA Confirmed</span>"
            title = "Structure validated as RNA"
            msg   = (f"Found RNA residues: <code>{fmt(info['rna'])}</code>. "
                     f"This file is compatible with oxRNA.")
            banner = ("<div class='sbanner'><span class='si'>&#10003;</span>"
                      "<span class='st'>oxRNA input configured</span></div>")

        dir_label = "3\u2032 \u2192 5\u2032" if direction == "35" else "5\u2032 \u2192 3\u2032"

        html = CSS + f"""
<div class='result-box'>
  {tag}
  <h2>{title}</h2>
  <p class='msg'>{msg}</p>
  <div class='cmd-block'>
    <span class='clabel'>oxRNA Arguments</span>
    <div class='arow'><span class='akey'>Script</span>
      <span class='aval'>pdb_oxDNA.py</span></div>
    <div class='arow'><span class='akey'>PDB file path</span>
      <span class='aval'>{path}</span></div>
    <div class='arow'><span class='akey'>Direction flag</span>
      <span class='aval'>{direction} &nbsp;&#8212;&nbsp; {dir_label}</span></div>
    <div class='arow'><span class='akey'>Models-as-strands (-m)</span>
      <span class='aval'>{'yes' if use_models else 'no (omit -m)'}</span></div>
    <div class='arow'><span class='akey'>Output topology (.top)</span>
      <span class='aval'>{top_file}</span></div>
    <div class='arow'><span class='akey'>Output configuration (.dat)</span>
      <span class='aval'>{dat_file} &nbsp;<span style='color:#475569;font-size:11px'>(saved as {oxdna_file})</span></span></div>
    <div class='fcmd'>
      <span class='fl'>Full command</span>{full_cmd}
    </div>
  </div>
  {banner}
</div>"""
        display(HTML(html))

btn.on_click(on_configure)